# 3.01 行为克隆与 DAgger

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/03_imitation/3.01_behavior_cloning.ipynb)


## 为什么要学

你在 ALOHA 上采集遥操作数据、微调 π0.5，这个过程就是**行为克隆**：把"观测 → 专家动作"当作监督学习问题。它简单、稳定、可扩展，是所有 VLA 的训练基础。

但 BC 有一个根本缺陷：**训练时只见过专家走过的状态，部署时却会走到自己的错误导致的新状态**。本课亲手复现这个问题，并用 DAgger（以及它的"人工接管"版本 HG-DAgger）解决它。这正是 RLT、HIL-SERL 等真机方法收集"人工干预数据"的理论基础。

---

## 1. 行为克隆 = 监督学习

给定示教数据集 $\mathcal D=\{(o_i, a_i)\}$，学习策略 $\pi_\theta(a|o)$：

$$
\min_\theta\ \mathbb{E}_{(o,a)\sim\mathcal D}\big[-\log\pi_\theta(a|o)\big]\quad\xrightarrow{\text{高斯假设}}\quad\min_\theta\ \mathbb{E}\big[\|\mu_\theta(o)-a\|^2\big]
$$

（1.05 课讲过：最大似然 + 固定方差的高斯 = MSE。）

### 环境：二维点到达

观测 $o=[x,y,g_x,g_y]$（当前位置 + 目标位置），动作 $a\in[-1,1]^2$ 是速度方向，每步移动 $0.05a$。到达目标 3cm 以内算成功（我们把它当作一个"精细定位"任务）。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.envs import PointReachEnv, collect_demos
from vla_ladder.utils import set_seed

RADIUS = 0.03            # 成功半径：越小越"精细"
env = PointReachEnv(num_envs=10, seed=0, success_radius=RADIUS)
demos = collect_demos(env, n_episodes=10, noise=0.0, seed=0)
print(f"{len(demos)} 条示教，共 {sum(len(d['obs']) for d in demos)} 帧")

plt.figure(figsize=(4, 4))
for d in demos:
    plt.plot(d["obs"][:, 0], d["obs"][:, 1], "-", lw=1)
    plt.plot(d["obs"][0, 2], d["obs"][0, 3], "r*")
plt.xlim(-1, 1); plt.ylim(-1, 1); plt.gca().set_aspect("equal"); plt.title("expert demos (red star = goal)"); plt.show()

### 练习 1：训练 BC 策略

把示教拼成数组，用 MSE 训练一个 MLP。

In [ ]:
def make_policy():
    return nn.Sequential(nn.Linear(4, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

def train_bc(obs, actions, steps=2000, batch_size=256, lr=1e-3, seed=0):
    """obs: (N, 4) numpy，actions: (N, 2) numpy -> 训练好的策略"""
    set_seed(seed)
    policy = make_policy()
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    O, A = torch.as_tensor(obs), torch.as_tensor(actions)
    for step in range(steps):
        # TODO: 随机采一个小批量（torch.randint），计算 MSE，更新参数
        raise NotImplementedError
    return policy

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def make_policy():
    return nn.Sequential(nn.Linear(4, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))

def train_bc(obs, actions, steps=2000, batch_size=256, lr=1e-3, seed=0):
    """obs: (N, 4) numpy，actions: (N, 2) numpy -> 训练好的策略"""
    set_seed(seed)
    policy = make_policy()
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    O, A = torch.as_tensor(obs), torch.as_tensor(actions)
    for step in range(steps):
        idx = torch.randint(0, len(O), (batch_size,))
        loss = ((policy(O[idx]) - A[idx]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return policy

### 练习 2：评估策略

在 200 个新的随机初始条件下运行策略，返回成功率。**评估一定要用训练时没见过的随机种子**。

In [ ]:
@torch.no_grad()
def evaluate(policy, n=200, seed=123, radius=RADIUS, return_trajs=False):
    env = PointReachEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = env.reset()
    trajs = [obs[:, :2].copy()]
    while not env.done.all():
        # TODO: 用策略算动作，env.step；把位置 obs[:, :2] 记录到 trajs
        raise NotImplementedError
    if return_trajs:
        return info["success"].mean(), np.stack(trajs, 1), env.goal.copy()
    return info["success"].mean()

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def evaluate(policy, n=200, seed=123, radius=RADIUS, return_trajs=False):
    env = PointReachEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = env.reset()
    trajs = [obs[:, :2].copy()]
    while not env.done.all():
        act = policy(torch.as_tensor(obs)).numpy()
        obs, _, _, info = env.step(act)
        trajs.append(obs[:, :2].copy())
    if return_trajs:
        return info["success"].mean(), np.stack(trajs, 1), env.goal.copy()
    return info["success"].mean()

In [ ]:
# ✅ 自测：示教越多，BC 越好
def stack(demos):
    return np.concatenate([d["obs"] for d in demos]), np.concatenate([d["actions"] for d in demos])

results = {}
for n_demo in [5, 10, 20, 50]:
    e = PointReachEnv(num_envs=n_demo, seed=0, success_radius=RADIUS)
    O, A = stack(collect_demos(e, n_demo, seed=0))
    results[n_demo] = (len(O), evaluate(train_bc(O, A)))
    print(f"{n_demo:3d} 条示教（{len(O):4d} 帧）-> 成功率 {results[n_demo][1]:.2f}")
assert results[50][1] > results[5][1]
print("通过 ✅")

看看少量数据训练出的策略**怎么失败**：

In [ ]:
O10, A10 = stack(demos)
pi10 = train_bc(O10, A10)
sr, trajs, goals = evaluate(pi10, n=12, return_trajs=True)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    ax.plot(trajs[i, :, 0], trajs[i, :, 1], "b.-", ms=2, lw=0.8)
    ax.add_patch(plt.Circle(goals[i], RADIUS, color="g", alpha=0.5))
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"BC with 10 demos (success {sr:.2f}): drift and oscillation near the goal"); plt.tight_layout(); plt.show()

---

## 2. 为什么会失败：复合误差与分布偏移

训练时，BC 只在**专家访问过的状态分布** $d_{\pi^*}$ 上被优化；部署时，策略面对的是**自己访问的状态分布** $d_{\pi_\theta}$。

一旦策略在某一步犯了一个小错（误差 $\epsilon$），它就进入一个专家没去过的状态，在那里它的误差可能更大，于是越走越偏……Ross & Bagnell (2010) 证明：

$$
\text{BC 的总代价} \le T\cdot J^* + O(T^2\epsilon)\qquad\text{（误差随任务长度 }T\text{ 二次增长）}
$$

而如果能在"策略自己访问的状态"上也得到专家标签，误差只会线性增长 $O(T\epsilon)$。

**对真机的启示**：

- 线头对插这种长时序、精度要求高的任务，误差累积尤其严重；
- 示教时**刻意加入"犯错-纠正"的片段**（比如故意偏一点再纠正回来），能显著提升鲁棒性；
- 动作分块（ACT/π0 一次输出 50 步）减少了"决策次数"，也缓解了复合误差（第 5 章）。

## 3. DAgger：在策略自己的状态上收集专家标签

**DAgger（Dataset Aggregation）** 算法：

```text
D ← 专家示教
重复 N 轮：
    π ← 在 D 上训练 BC
    用 π 与环境交互，记录它访问到的状态 s_1, s_2, ...
    请专家给这些状态打标签：a*_t = π*(s_t)
    D ← D ∪ {(s_t, a*_t)}
```

问题：真机上让人"给机器人自己走出来的每个状态标注应该怎么做"很难。**HG-DAgger**（Human-Gated DAgger）更实际：

- 机器人自己执行，**人类只在它快出错时接管**；
- 只记录**接管期间**的状态和人的动作；
- 接管后机器人回到正轨，继续执行。

这正是 RLT、HIL-SERL、FluxVLA 的 FluxDAgger 在真机上做的事：遥操作设备一直在旁边，操作员看到机器人要失败就按下接管键。

### 练习 3：模拟人工接管（HG-DAgger）

用脚本专家模拟"人"：当策略动作与专家动作的差距超过阈值 `thr` 时，"人"接管这一步（执行专家动作），并把 (状态, 专家动作) 加入数据集。

In [ ]:
@torch.no_grad()
def rollout_with_intervention(policy, n, seed, thr=0.3, radius=RADIUS):
    """返回 (接管时的观测 (M,4), 专家动作 (M,2), 接管比例)"""
    env = PointReachEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = env.reset()
    S, L, n_steps, n_interv = [], [], 0, 0
    while not env.done.all():
        alive = ~env.done
        a_pi = policy(torch.as_tensor(obs)).numpy()
        a_exp = env.expert_action()
        # TODO:
        # 1) intervene = alive & (||a_pi - a_exp|| > thr)
        # 2) 把 obs[intervene] 与 a_exp[intervene] 追加到 S、L
        # 3) 实际执行的动作：接管的环境用 a_exp，其余用 a_pi
        # 4) 统计 n_steps += alive.sum()，n_interv += intervene.sum()
        raise NotImplementedError
    return np.concatenate(S), np.concatenate(L), n_interv / max(n_steps, 1)

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def rollout_with_intervention(policy, n, seed, thr=0.3, radius=RADIUS):
    """返回 (接管时的观测 (M,4), 专家动作 (M,2), 接管比例)"""
    env = PointReachEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = env.reset()
    S, L, n_steps, n_interv = [], [], 0, 0
    while not env.done.all():
        alive = ~env.done
        a_pi = policy(torch.as_tensor(obs)).numpy()
        a_exp = env.expert_action()
        intervene = alive & (np.linalg.norm(a_pi - a_exp, axis=1) > thr)
        S.append(obs[intervene]); L.append(a_exp[intervene])
        act = np.where(intervene[:, None], a_exp, a_pi)
        n_steps += alive.sum(); n_interv += intervene.sum()
        obs, _, _, _ = env.step(act)
    return np.concatenate(S), np.concatenate(L), n_interv / max(n_steps, 1)

In [ ]:
# ✅ 自测：HG-DAgger 与"只加示教"的对比（看标注数量）
O, A = O10.copy(), A10.copy()
policy = train_bc(O, A)
hg_curve = [(len(O), evaluate(policy))]
for it in range(5):
    s, l, rate = rollout_with_intervention(policy, n=20, seed=1000 + it)
    O, A = np.concatenate([O, s]), np.concatenate([A, l])
    policy = train_bc(O, A)
    hg_curve.append((len(O), evaluate(policy)))
    print(f"第 {it + 1} 轮：接管比例 {rate:.2f}，新增 {len(s):3d} 条标注，累计 {len(O)} 条，成功率 {hg_curve[-1][1]:.2f}")

bc_curve = [results[k] for k in sorted(results)]
plt.figure(figsize=(5, 3.5))
plt.plot(*zip(*bc_curve), "o-", label="BC: more expert demos")
plt.plot(*zip(*hg_curve), "s-", label="HG-DAgger: interventions")
plt.xlabel("number of expert labels (frames)"); plt.ylabel("success rate"); plt.legend(); plt.grid(alpha=.3); plt.show()
assert hg_curve[-1][1] > 0.9
print("通过 ✅")

**结论**：同样多的专家标注，HG-DAgger 的效果好得多。因为它的标注**全部集中在策略真正会犯错的地方**，而普通示教中大部分帧是"策略本来就会做"的简单部分。注意接管比例也一轮轮下降，这在真机上意味着**操作员越来越轻松**。

---

## 小结

| 方法 | 数据来源 | 优点 | 缺点 |
|---|---|---|---|
| BC | 专家示教 | 简单稳定 | 分布偏移，复合误差 $O(T^2\epsilon)$ |
| DAgger | 策略访问的状态 + 专家标签 | 理论保证 $O(T\epsilon)$ | 需要专家给任意状态打标签 |
| HG-DAgger | 人只在要出错时接管 | 标注高效、真机可行 | 需要人一直盯着 |
| RL（第 8 章） | 策略自己试错 + 奖励 | 可以超越示教者 | 需要奖励信号、探索有成本 |

**思考题**：HG-DAgger 的数据里只有"出错时如何纠正"，没有"正常情况怎么做"。如果只用接管数据训练（不保留原始示教），会发生什么？（RLT 在训练 actor 时同时对"VLA 参考动作"和"人类接管动作"做 BC 正则，见 `rlt` 的 `bc_policy_row_weight` 配置。）